# 02 · Нейросеть FTW (PRUE) на ваших снимках через `ftw-tools`

**Что это.** Модели Fields of The World версии 3 (рецепт **PRUE**, U-Net с кодировщиком EfficientNet), которые
по двум сезонным снимкам Sentinel-2 (4 канала: красный, зелёный, синий, ближний ИК) выдают для каждого пикселя
вероятности «фон / поле / граница». Из них собираются полигоны полей. Это та же «семья» моделей, что построила
глобальную карту (ноутбук 01), но здесь вы запускаете её **на свой год** (например, 2026, которого в карте ещё нет)
и на **свои сезонные окна**.

**Что делает ноутбук.**
1. В GEE собирает **многодатный вход**: медиана Sentinel-2 за окно A (начало сезона) и окно B (вторая половина
   сезона), 8 каналов `B4,B3,B2,B8 (A) + B4,B3,B2,B8 (B)`, uint16, UTM 10 м. Большой участок режется на блоки.
2. Запускает `ftw inference run` на GPU (вероятности 0–255 по трём классам) и `ftw inference polygonize`.
3. Склеивает блоки, фильтрует полигоны по доле пашни WorldCover (как в конвейере), сохраняет GeoPackage.
4. Переносит вероятность **границы** на сетку вашего композита → GeoTIFF для гибрида с конвейером v7.
5. (по желанию) Сравнивает с вашей разметкой.

**Среда:** Среда выполнения → Сменить среду → **T4 GPU**. Время: экспорт GEE 5–30 мин на блок (идёт на серверах
Google), инференс ≈1–3 мин на блок 0,5°×0,5° на T4.

**Чего ждать (проверено на Донбассе, 2026, окна 10.04–31.05 и 15.07–31.08, модель B5, 301 эталонное поле).**
Сами полигоны слабые: IoU 0,29, точность 0,71, полнота 0,40; залежь почти не находится (охват 0,12), соседние
поля одного тона склеиваются. Но **карта границ** (шаг 6) полезна как барьеры в конвейере v7: v6 0,635 →
с границами PRUE 0,657; вместе с картой FTW Global (ноутбук 01) 0,678; ещё и с Higra — **0,684**.
На CPU блок 16×18 км считался ~2 мин; на T4 — в разы быстрее.

**Лицензии моделей.** `FTW_PRUE_EFNET_B5` (по умолчанию) обучена на смеси лицензий, включая некоммерческие (NC);
для коммерческого использования берите `…_CCBY` (обучены только на CC-BY/CC0 данных, чуть слабее).

## Шаг 0. Установка `ftw-tools` (один раз за сеанс)
Ячейка ставит `ftw-tools 2.0.0b5` (в ней есть модели PRUE; обычный `pip install ftw-tools` ставит старую 1.4.3 без них).
**После установки:** меню «Среда выполнения» → «Перезапустить сеанс», затем продолжайте **с шага 1** (шаг 0 повторно
запускать не нужно — он сам увидит, что всё уже стоит).

In [ ]:
import importlib.util, subprocess, sys
if importlib.util.find_spec('ftw_tools') is None:
    subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', 'ftw-tools[all]==2.0.0b5'], check=True)
    print('Установлено. Теперь: Среда выполнения → Перезапустить сеанс, и дальше с шага 1.')
else:
    import ftw_tools, torch
    print('ftw-tools уже установлен; torch', torch.__version__, '| GPU:', torch.cuda.is_available())

### Подключение Google Диска
Выполните ячейку и разрешите доступ к Диску (всплывёт окно Google). Все результаты пишутся в папку
`Colab_Notebooks/Agro_terra/research/…` — её можно поменять в параметрах ниже.

In [ ]:
import os, sys, json, time, glob, shutil, warnings
from pathlib import Path
warnings.filterwarnings('ignore')
LOCAL_TEST = bool(os.environ.get('LOCAL_TEST'))          # True только при проверке вне Colab
if not LOCAL_TEST:
    from google.colab import drive
    drive.mount('/content/drive')
    DRIVE_ROOT = Path('/content/drive/MyDrive')
else:
    DRIVE_ROOT = Path(os.environ['LOCAL_DRIVE'])
print('Диск:', DRIVE_ROOT, '| файлов в корне:', len(list(DRIVE_ROOT.iterdir())))

## Шаг 1. Параметры — заполните под себя

**Окна дат (A и B)** — главное, что нужно подобрать под регион. Модель ждёт снимок «в начале сезона» (озимые
зелёные, яровые только посеяны/всходы) и «во второй половине» (озимые убраны или желтеют, яровые в максимуме).
Ориентиры:

| регион | окно A | окно B |
|---|---|---|
| Донбасс, Ростов, Краснодар, Ставрополь, Волгоград | 10.04–31.05 | 15.07–31.08 |
| Тамбов, Липецк, Воронеж, Саратов, Самара | 20.04–10.06 | 20.07–05.09 |
| Татарстан, Ульяновск, Москва | 01.05–15.06 | 25.07–10.09 |
| Вологда, север Нечерноземья | 15.05–25.06 | 01.08–15.09 |

Если сомневаетесь — запустите два варианта окон на маленьком тестовом участке и сравните (шаг 7).

In [ ]:
EE_PROJECT = 'my-project-20260614-499410'

COMPOSITE_PATH = 'Colab_Notebooks/Agro_terra/TAT/TA_composite_2026_0504_0910_8b.tif'   # охват + сетка для вероятностей
AOI_PATH = None                     # или граница участка (geojson/gpkg)
BBOX = None                         # или (lon_min, lat_min, lon_max, lat_max) — удобно для быстрого теста
YEAR = 2026
WIN_A = ('2026-05-01', '2026-06-15')
WIN_B = ('2026-07-25', '2026-09-10')

MODEL = 'FTW_PRUE_EFNET_B5'         # или '…_B7' (точнее, медленнее), '…_B3', '…_B5_CCBY' (CC-BY), или путь к своему .ckpt из ноутбука 04
BLOCK_DEG = 0.5                     # размер блока экспорта/инференса в градусах
MIN_CROP_FRAC = 0.5                 # доля пашни WorldCover в полигоне (как в конвейере)
MIN_HA = 1.0                        # минимальная площадь поля, га
SIMPLIFY_M = 10                     # упрощение контуров при полигонизации, м

GT_PATH = None                      # ваша разметка для сравнения (необязательно)
GT_TYPE_COL = 'type'
OUT_DIR = 'Colab_Notebooks/Agro_terra/research/ftw_prue/out'
GEE_FOLDER = 'GEE_ftw_prue'         # временная папка в корне Диска для экспорта GEE
NAME = 'TAT_2026'

### Google Earth Engine: подключение и функции сборки многодатных композитов (общие для ноутбуков)
При первом запуске всплывёт окно авторизации Google — разрешите доступ для вашего проекта GEE.

Что умеют функции ниже:
- `s2_collection(aoi, start, end)` — Sentinel-2 L2A **HARMONIZED** (значения уже без смещения +1000, как ждут модели FTW),
  облака/тени маскируются по **Cloud Score+** (`cs_cdf ≥ CS_THR`);
- `window_median(aoi, start, end, bands)` — медиана за окно дат; пропуски (облака) заполняются медианой расширенного окна;
- `monthly_ndvi(aoi, year, months)` — помесячные медианы NDVI (многодатный стек для классических методов);
- `export_blocks(img, aoi, prefix, …)` — экспорт на Google Диск блоками (большой участок режется на квадраты с перекрытием);
- `wait_tasks(tasks)` — ждать окончания экспортов и показывать прогресс.

In [ ]:
import ee, math
try:
    ee.Initialize(project=EE_PROJECT)
except Exception:
    ee.Authenticate()
    ee.Initialize(project=EE_PROJECT)
print('GEE OK, проект', EE_PROJECT)

S2 = 'COPERNICUS/S2_SR_HARMONIZED'
CSP = 'GOOGLE/CLOUD_SCORE_PLUS/V1/S2_HARMONIZED'
CS_THR = 0.60          # порог Cloud Score+: 0.5 — мягче (больше данных), 0.65 — строже (меньше облаков)


def s2_collection(aoi, start, end, max_cloud=80):
    col = (ee.ImageCollection(S2).filterBounds(aoi).filterDate(start, end)
           .filter(ee.Filter.lt('CLOUDY_PIXEL_PERCENTAGE', max_cloud))
           .linkCollection(ee.ImageCollection(CSP), ['cs_cdf']))
    return col.map(lambda im: im.updateMask(im.select('cs_cdf').gte(CS_THR)))


def _shift(date, days):
    return ee.Date(date).advance(days, 'day')


def window_median(aoi, start, end, bands, pad_days=30):
    """Медиана за окно [start, end); дыры от облаков заполняются медианой окна, расширенного на ±pad_days."""
    core = s2_collection(aoi, start, end).select(bands).median()
    wide = s2_collection(aoi, _shift(start, -pad_days), _shift(end, pad_days)).select(bands).median()
    return core.unmask(wide)


def monthly_ndvi(aoi, year, months):
    """Многоканальный образ: NDVI-медиана каждого месяца (каналы ndvi_MM), 0–1 → ×10000 int16."""
    out = []
    for m in months:
        start = ee.Date.fromYMD(year, m, 1); end = start.advance(1, 'month')
        nd = s2_collection(aoi, start, end).map(lambda im: im.normalizedDifference(['B8', 'B4']).rename('ndvi')).median()
        out.append(nd.rename(f'ndvi_{m:02d}'))
    return ee.Image.cat(out).multiply(10000).round().toInt16()


def utm_epsg(lon, lat):
    return (32600 if lat >= 0 else 32700) + int((lon + 180) // 6) + 1


def block_grid(bbox, block_deg=0.5, overlap_deg=0.02):
    """Квадраты (по долготе/широте) с перекрытием: [(id, core_bbox, export_bbox)]."""
    x0, y0, x1, y1 = bbox; out = []
    nx, ny = max(1, math.ceil((x1 - x0) / block_deg)), max(1, math.ceil((y1 - y0) / block_deg))
    for i in range(nx):
        for j in range(ny):
            cx0, cy0 = x0 + i * block_deg, y0 + j * block_deg
            core = (cx0, cy0, min(cx0 + block_deg, x1), min(cy0 + block_deg, y1))
            exp = (core[0] - overlap_deg, core[1] - overlap_deg, core[2] + overlap_deg, core[3] + overlap_deg)
            out.append((f'b{i:02d}_{j:02d}', core, exp))
    return out


def export_blocks(img, bbox, prefix, gee_folder, scale=10, block_deg=0.5, dtype='uint16', crs=None):
    """Экспорт изображения на Диск блоками; возвращает список (id, core_bbox, имя_файла, задача).
    Файлы появятся в «Мой диск/<gee_folder>/<prefix>_<id>.tif». crs по умолчанию — UTM центра участка."""
    crs = crs or f'EPSG:{utm_epsg((bbox[0] + bbox[2]) / 2, (bbox[1] + bbox[3]) / 2)}'
    img = {'uint16': img.toUint16(), 'uint8': img.toUint8(), 'int16': img.toInt16(), 'float': img.toFloat()}[dtype]
    jobs = []
    for bid, core, exp in block_grid(bbox, block_deg):
        name = f'{prefix}_{bid}'
        task = ee.batch.Export.image.toDrive(image=img.clip(ee.Geometry.Rectangle(list(exp))), description=name[:100],
                                             folder=gee_folder, fileNamePrefix=name, region=ee.Geometry.Rectangle(list(exp)),
                                             scale=scale, crs=crs, maxPixels=1e13, fileFormat='GeoTIFF',
                                             fileDimensions=16384)     # один файл на блок (без нарезки на части)
        task.start(); jobs.append((bid, core, name, task))
    print(f'Запущено экспортов: {len(jobs)} (CRS {crs}); следить можно и на https://code.earthengine.google.com/tasks')
    return jobs


def wait_tasks(jobs, poll=30):
    """Ждёт завершения всех задач экспорта. Можно прервать ячейку и вернуться позже — задачи идут на серверах GEE."""
    t0 = time.time()
    while True:
        st = [j[3].status()['state'] for j in jobs]
        done = sum(s == 'COMPLETED' for s in st); bad = [j[2] for j, s in zip(jobs, st) if s in ('FAILED', 'CANCELLED')]
        print(f'\r{time.time() - t0:6.0f} с: готово {done}/{len(jobs)}' + (f', ошибки: {bad}' if bad else ''), end='')
        if done + len(bad) == len(jobs):
            print(); break
        time.sleep(poll)
    for j, s in zip(jobs, st):
        if s == 'FAILED':
            print(j[2], j[3].status().get('error_message'))


def collect_exports(jobs, gee_folder, dst_dir, timeout=900):
    """Переносит экспортированные файлы из «Мой диск/<gee_folder>» в папку проекта (Диск синхронизируется с задержкой)."""
    src_dir = DRIVE_ROOT / gee_folder; dst_dir = Path(dst_dir); dst_dir.mkdir(parents=True, exist_ok=True)
    t0 = time.time(); left = {j[2] for j in jobs}
    while left and time.time() - t0 < timeout:
        for name in list(left):
            for f in src_dir.glob(f'{name}*.tif'):                 # большой блок GEE может разрезать на части …-0000000000-0000000000.tif
                shutil.move(str(f), dst_dir / f.name)
            if list(dst_dir.glob(f'{name}*.tif')):
                left.discard(name)
        if left:
            time.sleep(15)
    print('Файлы в', dst_dir, ':', len(list(dst_dir.glob('*.tif'))), '| не найдено:', sorted(left))

## Шаг 2. Участок и блоки

In [ ]:
import numpy as np, pandas as pd, geopandas as gpd, shapely, rasterio
from rasterio.warp import transform_bounds
OUT = DRIVE_ROOT / OUT_DIR; OUT.mkdir(parents=True, exist_ok=True)
if MODEL.endswith('.ckpt'):                                    # своя модель из ноутбука 04: путь от корня Диска или полный
    MODEL_PATH = MODEL if MODEL.startswith('/') else str(DRIVE_ROOT / MODEL); MODEL_TAG = Path(MODEL).stem
else:
    MODEL_PATH = MODEL_TAG = MODEL
IMG_DIR = OUT / f'{NAME}_input'; INF_DIR = OUT / f'{NAME}_{MODEL_TAG}_inference'; INF_DIR.mkdir(parents=True, exist_ok=True)
if BBOX:
    AOI_BBOX = tuple(BBOX)
elif COMPOSITE_PATH:
    with rasterio.open(DRIVE_ROOT / COMPOSITE_PATH) as s:
        AOI_BBOX = transform_bounds(s.crs, 'EPSG:4326', *s.bounds)
else:
    AOI_BBOX = tuple(gpd.read_file(DRIVE_ROOT / AOI_PATH).to_crs(4326).total_bounds)
BLOCKS = block_grid(AOI_BBOX, BLOCK_DEG)
print('рамка', np.round(AOI_BBOX, 3), '| блоков:', len(BLOCKS))

## Шаг 3. Многодатный вход в GEE: окно A + окно B → 8 каналов, экспорт на Диск
Для каждого окна берётся медиана безоблачных пикселей (маска Cloud Score+); дыры заполняются медианой окна,
расширенного на ±30 дней. Если файлы блоков уже лежат в `<NAME>_input` (например, после прерванного сеанса),
экспорт этих блоков пропускается.

Ячейка ждёт окончания экспортов (видно «готово N из M»). Её можно прервать — задачи продолжатся на серверах GEE;
тогда потом просто запустите её снова: готовые блоки не будут экспортироваться повторно.

In [ ]:
aoi_ee = ee.Geometry.Rectangle(list(AOI_BBOX))
BANDS = ['B4', 'B3', 'B2', 'B8']
img = (window_median(aoi_ee, WIN_A[0], WIN_A[1], BANDS).rename([f'{b}_A' for b in BANDS])
       .addBands(window_median(aoi_ee, WIN_B[0], WIN_B[1], BANDS).rename([f'{b}_B' for b in BANDS])))
todo = [b for b in BLOCKS if not list(IMG_DIR.glob(f'{NAME}_{b[0]}*.tif'))]
if todo:
    jobs = []
    crs = f'EPSG:{utm_epsg((AOI_BBOX[0] + AOI_BBOX[2]) / 2, (AOI_BBOX[1] + AOI_BBOX[3]) / 2)}'
    for bid, core, exp in todo:
        name = f'{NAME}_{bid}'; reg = ee.Geometry.Rectangle(list(exp))
        t = ee.batch.Export.image.toDrive(image=img.toUint16().clip(reg), description=name, folder=GEE_FOLDER,
                                          fileNamePrefix=name, region=reg, scale=10, crs=crs, maxPixels=1e13,
                                          fileDimensions=16384)
        t.start(); jobs.append((bid, core, name, t))
    wait_tasks(jobs)
    collect_exports(jobs, GEE_FOLDER, IMG_DIR)
else:
    print('Все блоки уже экспортированы:', len(BLOCKS))

## Шаг 4. Инференс FTW и полигоны
Для каждого блока: `ftw inference run --save_scores` (3 канала вероятностей ×255: фон, поле, граница) и
`ftw inference polygonize --softmax_threshold 0.5`. Готовые блоки пропускаются (можно перезапускать).
Первый запуск скачивает веса модели (~100–300 МБ).

In [ ]:
import subprocess, torch
GPU = 0 if torch.cuda.is_available() else -1
print('GPU' if GPU == 0 else 'ВНИМАНИЕ: GPU нет, будет медленно (CPU)')
for f in sorted(IMG_DIR.glob(f'{NAME}_b*.tif')):
    scores = INF_DIR / f'{f.stem}_scores.tif'; poly = INF_DIR / f'{f.stem}_fields.gpkg'
    if not scores.exists():
        r = subprocess.run(['ftw', 'inference', 'run', str(f), '--model', MODEL_PATH, '--out', str(scores), '--gpu', str(GPU),
                            '--save_scores', '--num_workers', '2', '--overwrite'], capture_output=True, text=True)
        if r.returncode:
            print(r.stdout[-2000:], r.stderr[-3000:]); raise SystemExit('ошибка инференса ' + f.name)
    if not poly.exists():
        r = subprocess.run(['ftw', 'inference', 'polygonize', str(scores), '--out', str(poly), '--softmax_threshold', '0.5',
                            '--simplify', str(SIMPLIFY_M), '--min_size', str(int(MIN_HA * 1e4 / 2)), '--overwrite'],
                           capture_output=True, text=True)
        if r.returncode:
            print(r.stdout[-2000:], r.stderr[-3000:]); raise SystemExit('ошибка полигонизации ' + f.name)
    print('готов блок', f.stem)

### Постобработка (общая): доля пашни по ESA WorldCover и перенос растров на сетку композита
- `cropland_frac(gdf)` — доля пикселей WorldCover-«пашни» (класс 40) внутри каждого полигона; читается прямо из
  открытого хранилища ESA (AWS), GEE не нужен;
- `filter_fields(gdf, MIN_CROP_FRAC, MIN_HA)` — как в конвейере: оставляет полигоны, где пашни ≥ 50 % и площадь ≥ MIN_HA;
- `to_grid(src_tif, band, composite)` — переносит канал растра (например, вероятность границы) на сетку вашего композита.

In [ ]:
import math
from rasterio.warp import reproject, Resampling, transform_bounds
from rasterio.windows import from_bounds
from rasterio.features import rasterize


def _wc_tiles(bbox):
    x0, y0, x1, y1 = bbox
    for lat in range(int(math.floor(y0 / 3) * 3), int(math.floor(y1 / 3) * 3) + 1, 3):
        for lon in range(int(math.floor(x0 / 3) * 3), int(math.floor(x1 / 3) * 3) + 1, 3):
            ns, ew = ('N' if lat >= 0 else 'S'), ('E' if lon >= 0 else 'W')
            yield (f'/vsicurl/https://esa-worldcover.s3.eu-central-1.amazonaws.com/v200/2021/map/'
                   f'ESA_WorldCover_10m_2021_v200_{ns}{abs(lat):02d}{ew}{abs(lon):03d}_Map.tif')


def cropland_frac(gdf, agri=(40,)):
    """Доля пашни WorldCover 2021 в каждом полигоне (0–1). gdf — любая СК."""
    g = gdf.to_crs(4326); frac = np.zeros(len(g)); tot = np.zeros(len(g))
    if not len(g):
        return frac
    bbox = tuple(g.total_bounds)
    with rasterio.Env(GDAL_DISABLE_READDIR_ON_OPEN='EMPTY_DIR', CPL_VSIL_CURL_ALLOWED_EXTENSIONS='.tif'):
        for url in _wc_tiles(bbox):
            try:
                s = rasterio.open(url)
            except Exception:
                continue
            with s:
                b = (max(bbox[0], s.bounds.left), max(bbox[1], s.bounds.bottom), min(bbox[2], s.bounds.right), min(bbox[3], s.bounds.top))
                if b[0] >= b[2] or b[1] >= b[3]:
                    continue
                win = from_bounds(*b, s.transform).round_offsets().round_lengths()
                a = s.read(1, window=win); t = s.window_transform(win)
            ids = rasterize([(geom, i + 1) for i, geom in enumerate(g.geometry.values)], out_shape=a.shape, transform=t,
                            fill=0, dtype='int32')
            n = len(g) + 1
            tot += np.bincount(ids.ravel(), minlength=n)[1:]
            frac += np.bincount(ids.ravel(), weights=np.isin(a, agri).ravel().astype(float), minlength=n)[1:]
    return np.where(tot > 0, frac / np.maximum(tot, 1), 0.0)


def filter_fields(gdf, min_crop_frac=0.5, min_ha=1.0):
    g = gdf.copy()
    g['area_ha'] = g.to_crs(6933).area / 1e4
    g['crop_frac'] = cropland_frac(g)
    keep = (g.crop_frac >= min_crop_frac) & (g.area_ha >= min_ha)
    print(f'фильтр: осталось {keep.sum()} из {len(g)} (пашня ≥ {min_crop_frac:.0%}, площадь ≥ {min_ha} га)')
    return g[keep].reset_index(drop=True)


def to_grid(src_tif, band, composite, resampling=Resampling.max):
    """Канал растра → массив на сетке композита (float32, NaN — нет данных)."""
    with rasterio.open(composite) as c:
        out = np.full((c.height, c.width), np.nan, np.float32); T, crs = c.transform, c.crs
    with rasterio.open(src_tif) as s:
        reproject(rasterio.band(s, band), out, dst_transform=T, dst_crs=crs, dst_nodata=np.nan, resampling=resampling)
    return out

## Шаг 5. Склейка блоков, фильтр по пашне, сохранение
Полигон относится к тому блоку, в «ядро» которого попадает его центр (перекрытие блоков убирает разрезы по шву).

In [ ]:
core_of = {f'{NAME}_{bid}': core for bid, core, _ in BLOCKS}
parts = []
for p in sorted(INF_DIR.glob(f'{NAME}_b*_fields.gpkg')):
    g = gpd.read_file(p); key = p.stem.replace('_fields', '').split('-')[0]
    core = core_of.get(key)
    if core is not None and len(g):
        c = g.to_crs(4326).representative_point()
        g = g[c.within(shapely.box(*core)).values]
    parts.append(g.to_crs(4326))
fields = gpd.GeoDataFrame(pd.concat(parts, ignore_index=True), crs=4326)
if AOI_PATH:
    fields = fields[fields.intersects(gpd.read_file(DRIVE_ROOT / AOI_PATH).to_crs(4326).union_all())]
print('всего полигонов до фильтра:', len(fields))
fields = filter_fields(fields, MIN_CROP_FRAC, MIN_HA)
out_gpkg = OUT / f'{NAME}_{MODEL_TAG}_fields.gpkg'
fields.to_file(out_gpkg, layer='fields', driver='GPKG')
print('Сохранено:', out_gpkg, '| медиана площади', round(fields.area_ha.median(), 1), 'га')

## Шаг 6. Вероятность границы на сетку композита (для гибрида с конвейером v7)
Пишется `<NAME>_<MODEL>_boundary.tif` (uint8, 1–255; 0 — нет данных) на сетке `COMPOSITE_PATH`. В конвейере v7
укажите его в `EXT_BOUNDARY` (вместе с картой FTW Global — списком) — линии границ нейросети станут дополнительными барьерами при слиянии.

In [ ]:
if COMPOSITE_PATH:
    with rasterio.open(DRIVE_ROOT / COMPOSITE_PATH) as c:
        prof = dict(driver='GTiff', width=c.width, height=c.height, count=1, dtype='uint8', crs=c.crs, transform=c.transform,
                    compress='deflate', tiled=True, BIGTIFF='IF_SAFER', nodata=0)
        acc = np.zeros((c.height, c.width), np.uint8)
    for sc in sorted(INF_DIR.glob(f'{NAME}_b*_scores.tif')):
        b = to_grid(sc, 3, DRIVE_ROOT / COMPOSITE_PATH)          # канал 3 = граница
        ok = ~np.isnan(b); acc[ok] = np.maximum(acc[ok], np.clip(b[ok] * 254 / 255 + 1, 1, 255).astype(np.uint8))
    bnd_tif = OUT / f'{NAME}_{MODEL_TAG}_boundary.tif'
    with rasterio.open(bnd_tif, 'w', **prof) as d:
        d.write(acc, 1)
    print('Сохранено:', bnd_tif)

### Функции оценки и картинок (общие для всех ноутбуков)
Ничего менять не нужно — просто выполните ячейку. Метрики те же, что в тестах конвейера
(`tests/eval_gt.py` в репозитории), поэтому числа можно сравнивать с v6:
- **IoU** — для каждого эталонного поля (≥ 5 га) берётся лучший по перекрытию полигон, среднее по полям;
- **IoU≥0.7** — доля эталонных полей, найденных «хорошо»;
- **frag** — сколько полигонов в среднем покрывают одно эталонное поле (1 — идеально, >1 — дробление);
- **merged** — сколько полигонов «склеили» 2 и больше эталонных полей;
- **bF** — точность границ (F-мера, допуск 2 пикселя), **P/R** — точность/полнота линий;
- **охват** — доля площади эталона, покрытая полигонами; по классам (если в эталоне есть колонка `type`).

In [ ]:
import numpy as np, geopandas as gpd, shapely, rasterio, rasterio.windows
from rasterio.features import rasterize
from rasterio.transform import from_origin
from scipy import ndimage as ndi


def load_gt(path, bbox=None):
    """Эталонная разметка: только полигоны в рамке bbox (EPSG:4326: lon_min, lat_min, lon_max, lat_max), без пустых
    и повторных. Так можно указывать разметку на всю область (например DNR_type_2026.geojson), а сравнивать на участке."""
    box = gpd.GeoSeries([shapely.box(*bbox)], crs=4326) if bbox is not None else None
    g = gpd.read_file(path, bbox=box)
    g = g[g.geometry.notna()]
    if bbox is not None:
        g = g[g.to_crs(4326).representative_point().within(box.iloc[0]).values]
    g = g[~g.geometry.to_wkb().duplicated()].reset_index(drop=True)
    print('эталон:', len(g), 'полигонов', '(в рамке участка)' if bbox is not None else '')
    return g


def grid_for(gdf, res=10.0):
    """Сетка 10 м в UTM по охвату слоя (если нет вашего композита)."""
    utm = gdf.estimate_utm_crs()
    x0, y0, x1, y1 = gdf.to_crs(utm).total_bounds
    w, h = int(np.ceil((x1 - x0) / res)) + 2, int(np.ceil((y1 - y0) / res)) + 2
    return from_origin(x0 - res, y1 + res, res, res), (h, w), utm


def evaluate(pred, gt, transform=None, shape=None, crs=None, min_ha=5, type_col='type', label=''):
    """pred, gt — GeoDataFrame полигонов (любая СК). transform/shape/crs — сетка для растровых метрик
    (по умолчанию 10 м UTM по эталону). Печатает строку метрик и возвращает dict."""
    if transform is None:
        transform, shape, crs = grid_for(gt)
    H, W = shape
    gt = gt[gt.geometry.notna()].copy(); pred = pred[pred.geometry.notna()].copy()
    gt = gt[~gt.geometry.to_wkb().duplicated()]                # в выгрузках разметки бывают повторы одного полигона
    box = shapely.box(*rasterio.transform.array_bounds(H, W, transform))
    gt = gt[gt.to_crs(crs).representative_point().within(box).values]   # эталон — только в пределах сетки (участка)
    if not len(gt):
        print(label, ': в пределах участка нет эталонных полей'); return dict(label=label, gt=0)
    win = rasterio.windows.from_bounds(*gt.to_crs(crs).total_bounds, transform)          # сетка — только охват эталона
    win = win.round_offsets().round_lengths()
    c0, r0 = max(int(win.col_off) - 20, 0), max(int(win.row_off) - 20, 0)
    W, H = min(int(win.col_off + win.width) + 20, W) - c0, min(int(win.row_off + win.height) + 20, H) - r0
    transform = rasterio.windows.transform(rasterio.windows.Window(c0, r0, W, H), transform)
    if H * W > 4e8:
        print(f'ВНИМАНИЕ: эталон охватывает {H}×{W} пикселей — оценка займёт много памяти; задайте участок поменьше (BBOX)')
    gt['geometry'] = shapely.make_valid(gt.geometry.values); pred['geometry'] = shapely.make_valid(pred.geometry.values)
    ea = 6933                                                  # равновеликая СК для площадей
    g6 = gt.to_crs(ea); g6 = g6[g6.area / 1e4 >= min_ha].reset_index(drop=True)
    p6 = pred.to_crs(ea); p6['geometry'] = shapely.make_valid(p6.geometry.values)
    tree = shapely.STRtree(p6.geometry.values)
    ious, frag, merged = [], [], np.zeros(len(p6), int)
    for g in g6.geometry.values:
        best, nfr = 0.0, 0
        for j in tree.query(g, predicate='intersects'):
            pj = p6.geometry.values[j]; inter = g.intersection(pj).area
            if inter >= .15 * g.area: merged[j] += 1
            if inter >= .05 * g.area: nfr += 1
            best = max(best, inter / g.union(pj).area)
        ious.append(best); frag.append(nfr)
    ious = np.array(ious)
    idmap = lambda d: rasterize([(g, i + 1) for i, g in enumerate(d.to_crs(crs).geometry.values)],
                                out_shape=(H, W), transform=transform, fill=0, dtype='int32')
    GT, P = idmap(g6), idmap(p6)
    def bnd(m):
        b = np.zeros(m.shape, bool); b[:, :-1] |= m[:, :-1] != m[:, 1:]; b[:-1] |= m[:-1] != m[1:]; return b
    gb, pb = bnd(GT) & (GT > 0), bnd(P) & (P > 0)
    dgt, dpb = ndi.distance_transform_edt(~gb), ndi.distance_transform_edt(~pb)
    dom = ndi.binary_dilation(GT > 0, iterations=3)
    prec = (dgt[pb & dom] <= 2).mean() if (pb & dom).any() else 0.0
    rec = (dpb[gb] <= 2).mean() if gb.any() else 0.0
    bF = 2 * prec * rec / (prec + rec + 1e-9)
    cov = ((P > 0) & (GT > 0)).sum() / max((GT > 0).sum(), 1)
    res = dict(label=label, gt=len(g6), pred=len(pred), IoU=round(float(ious.mean()), 3) if len(ious) else 0,
               IoU07=round(float(np.mean(ious >= .7)), 2) if len(ious) else 0, frag=round(float(np.mean(frag)), 2),
               merged=int((merged >= 2).sum()), bF=round(float(bF), 3), P=round(float(prec), 2), R=round(float(rec), 2),
               cover=round(float(cov), 2))
    if type_col in gt.columns:
        gtt = gt.to_crs(crs)
        for t in sorted(gtt[type_col].dropna().unique()):
            m = rasterize([(g, 1) for g in gtt[gtt[type_col] == t].geometry.values], out_shape=(H, W),
                          transform=transform, fill=0, dtype='uint8') > 0
            res[f'cover_{t}'] = round(float((P[m] > 0).mean()), 2) if m.any() else None
    print(' | '.join(f'{k} {v}' for k, v in res.items()))
    return res


def show_overlay(img_path, layers, title='', bbox=None, out_png=None, bands=(1, 2, 3), figsize=(11, 11)):
    """Картинка: подложка — ваш растр (3 канала, растяжка 2–98 %), поверх контуры слоёв.
    layers — список (GeoDataFrame, цвет, подпись)."""
    import matplotlib.pyplot as plt
    from rasterio.windows import from_bounds
    with rasterio.open(img_path) as s:
        win = from_bounds(*bbox, s.transform) if bbox is not None else rasterio.windows.Window(0, 0, s.width, s.height)
        k = max(1.0, max(win.width, win.height) / 2500)                  # большие охваты читаются с прореживанием
        shp = (len(bands), int(win.height / k), int(win.width / k))
        a = s.read(list(bands), window=win, boundless=True, out_shape=shp).astype(float)
        t = s.window_transform(win) * rasterio.Affine.scale(win.width / shp[2], win.height / shp[1]); crs = s.crs
    for i in range(a.shape[0]):
        lo, hi = np.nanpercentile(a[i][a[i] > 0], [2, 98]) if (a[i] > 0).any() else (0, 1)
        a[i] = np.clip((a[i] - lo) / (hi - lo + 1e-9), 0, 1)
    ext = (t.c, t.c + t.a * a.shape[2], t.f + t.e * a.shape[1], t.f)
    fig, ax = plt.subplots(figsize=figsize)
    ax.imshow(np.moveaxis(a, 0, -1), extent=ext)
    for gdf, color, name in layers:
        if gdf is not None and len(gdf):
            gdf.to_crs(crs).boundary.plot(ax=ax, color=color, linewidth=.8, label=name)
    ax.set_xlim(ext[0], ext[1]); ax.set_ylim(ext[2], ext[3]); ax.set_title(title); ax.legend(loc='upper right')
    if out_png:
        fig.savefig(out_png, dpi=130, bbox_inches='tight')
    plt.show()

## Шаг 7 (по желанию). Сравнение с разметкой и картинка

In [ ]:
if GT_PATH:
    gt = load_gt(DRIVE_ROOT / GT_PATH, AOI_BBOX)
    kw = {}
    if COMPOSITE_PATH:
        with rasterio.open(DRIVE_ROOT / COMPOSITE_PATH) as c:
            kw = dict(transform=c.transform, shape=(c.height, c.width), crs=c.crs)
    res = evaluate(fields[fields.intersects(shapely.box(*gt.to_crs(4326).total_bounds))], gt, type_col=GT_TYPE_COL,
                   label=f'{MODEL_TAG} {YEAR}', **kw)
    pd.DataFrame([res]).to_csv(OUT / f'{NAME}_{MODEL_TAG}_vs_gt.csv', index=False)
    if COMPOSITE_PATH:
        with rasterio.open(DRIVE_ROOT / COMPOSITE_PATH) as c:
            vb = transform_bounds('EPSG:4326', c.crs, *gt.to_crs(4326).total_bounds)
        show_overlay(DRIVE_ROOT / COMPOSITE_PATH, [(fields, 'cyan', MODEL_TAG), (gt, 'red', 'эталон')], title=NAME, bbox=vb,
                     out_png=OUT / f'{NAME}_{MODEL_TAG}_view.png')

## Что дальше
- Сравните с картой FTW (ноутбук 01) и с конвейером v6 на одном и том же участке.
- Если поля дробятся по культурам, а залежь пропускается — это ожидаемо: модель обучена на декларациях ЕС.
  Лечится либо гибридом (шаг 6 → конвейер v7), либо дообучением на ваших полях (ноутбук 04).
- Свою дообученную модель (файл `.ckpt` из ноутбука 04) можно подставить в `MODEL` путём от корня Диска, например
  `MODEL = 'Colab_Notebooks/Agro_terra/research/ftw_finetune/out/ru_v1_FTW_PRUE_EFNET_B3_CCBY.ckpt'`.